In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

sentences = [
    ("Eu amo esse filme", 1),
    ("Esse filme é ótimo", 1),
    ("Experiência Incrível", 1),
    ("Eu odeio esse filme", 0),
    ("Esse filme é terrível", 0),
    ("Experiência horrível", 0)
]

In [3]:
# Função básica para separar a frase em palavras minúsculas
def tokenize(sentence):
    return sentence.lower().split()

# Construindo vocabulário om token de segurança para palavras desconhecidas
UNK_TOKEN = 1
vocab = {"<UNK>" : UNK_TOKEN}

# Preenchendo o vocabulário com as palavras do dataset de treino
for sentence, _ in sentences:
    for word in tokenize(sentence):
        if word not in vocab:
            vocab[word] = len(vocab) + 1 # IDs começam do 2 em diante

# Tamanho do vocabulário (+1 para considerar o 0 que será usado no Padding)
vocab_size = len(vocab) + 1

# Função que converte as palavras nos seus respectivos números
def encode(sentence):
    # Se a palavra existir no vocab, retorna o ID dela. Se não, retorna o UNK_TOKEN (1)
    return [vocab.get(word, UNK_TOKEN) for word in tokenize(sentence)]

encoded_data = [(encode(s), label) for s, label in sentences]

# Função para aplicar Padding (adicionar zeros)
def pad_sequence(seq, max_len):
    return seq + [0] * (max_len - len(seq))

# Descobrimos qual é a frase mais longa do nosso dataset
max_len = max(len(seq) for seq, _ in encoded_data)

# Criando os tensores finais (X = dados de entrada, y = rótulos/respostas)
X = torch.tensor([pad_sequence(seq, max_len) for seq, _ in encoded_data])
y = torch.tensor([label for _, label in encoded_data], dtype=torch.float32)

print("Tensores: ")

print("="*50)
print("Entradas:")
print(X)

print("="*50)
print("Rótulos:")
print(y)

Tensores: 
Entradas:
tensor([[ 2,  3,  4,  5],
        [ 4,  5,  6,  7],
        [ 8,  9,  0,  0],
        [ 2, 10,  4,  5],
        [ 4,  5,  6, 11],
        [ 8, 12,  0,  0]])
Rótulos:
tensor([1., 1., 1., 0., 0., 0.])


In [4]:
class SentimentRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super(SentimentRNN, self).__init__()

        # Camada que mapeia palavras para vetores
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # A Vanilla RNN, batch_first=True indica que o tensor de entrada
        # terá o formato (tamanho_do_lote, tamanho_da_sequencia, dimensão_do_embedding)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)

        # Camada de classificação final
        self.fc = nn.Linear(hidden_dim, 1)

        # Função de ativação para gerar uma probabilidade (0 a 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # 1. Passa as palavras pela camada de Embedding
        x = self.embedding(x)

        # 2. Passa pela RNN. Uma RNN simples retorna as saídas de todos os passos 
        # e o último estado oculto (hidden_state).
        out_rnn,hidden = self.rnn(x)

        # 3. Queremos apenas o último estado oculto (-1) para a classificação
        out = self.fc(hidden[-1])

        # 4. Retorna a probabilidade
        return self.sigmoid(out)

# Instanciando nosso modelocom hiperparâmetros de exemplo
embedding_dim = 10
hidden_dim = 16

model = SentimentRNN(vocab_size, embedding_dim, hidden_dim)

In [5]:
criterion = nn.BCELoss()  # Função de perda para classificação binária
optimizer = optim.Adam(model.parameters(), lr=0.01)  # Algoritmo de otimização

epochs = 100

for epoch in range(epochs):
    # Coloca o modelo em modo de treinamento
    model.train()

    # Forward pass: passa os dados pelo modelo
    outputs = model(X).squeeze()

    # Calcula a perda
    loss = criterion(outputs, y)

    # Backward pass: calcula os gradientes e atualiza os pesos
    optimizer.zero_grad()  # Zera gradientes antigos
    loss.backward()        # Calcula o erro de trás para frente
    optimizer.step()       # Atualiza os pesos

    # Mostra a perda a cada 10 épocas
    if (epoch + 1) % 10 == 0:
        print(f"Época [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Época [10/100], Loss: 0.4215
Época [20/100], Loss: 0.0759
Época [30/100], Loss: 0.0132
Época [40/100], Loss: 0.0052
Época [50/100], Loss: 0.0031
Época [60/100], Loss: 0.0023
Época [70/100], Loss: 0.0019
Época [80/100], Loss: 0.0017
Época [90/100], Loss: 0.0015
Época [100/100], Loss: 0.0013


In [6]:
def predict(sentence):
    # Coloca o modelo em modo de avaliação
    model.eval()

    # Pré-processamento idêntico ao treino
    seq = encode(sentence)
    seq = pad_sequence(seq, max_len)
    seq = torch.tensor([seq]) # Adiciona dimensão de batch artificial

    # Desativa cálculo de gradientes
    with torch.no_grad():
        output = model(seq).item()

    # Define a fronteira de decisão: maior que 0.5 é Positivo, senão Negativo
    return f"Positivo {output:.15f}" if output > 0.5 else f"Negativo {output:.15f}"

In [7]:
# Testando com frases conhecidas e desconhecidas
print("\nTestes:")
print("Eu amo esse filme ->", predict("Eu amo esse filme"))
print("Esse filme é terrível ->", predict("Esse filme é terrível"))

# Teste da OOV (Palvras não conhecidas pelo vocabulário)
print(
    "O filme do Silvio Santos é horrível ->",
    predict("O filme do Silvio Santos é horrível"),
)
print("O filme do Batman é ótimo ->", predict("O filme do Batman é ótimo"))


Testes:
Eu amo esse filme -> Positivo 0.999082207679749
Esse filme é terrível -> Negativo 0.002135662361979
O filme do Silvio Santos é horrível -> Positivo 0.656026124954224
O filme do Batman é ótimo -> Positivo 0.998634278774261
